In [3]:
import urllib.request
import os

os.makedirs('docs', exist_ok=True)

url = "https://www.plant-maintenance.com/articles/hydraulic_fluid_cleanliness.pdf"
output_path = "docs/maintenance_guide.pdf"

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

req = urllib.request.Request(url, headers=headers)

with urllib.request.urlopen(req) as response:
    with open(output_path, 'wb') as f:
        f.write(response.read())

print("Done.")

Done.


# RAG Pipeline — Maintenance Document Q&A

**Goal:** Build a system that answers questions over maintenance documents
using retrieval-augmented generation.

**Pipeline:** Load docs → Chunk → Embed → Store in FAISS → Retrieve → Generate

**Why RAG:** The LLM has no knowledge of our specific equipment.
RAG grounds answers in our actual documentation, reducing hallucination
and making answers auditable, you can show the source passage.

## Load and Chunk

In [1]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load all documents from the docs folder
docs_path = Path('docs')
documents = []

for file in docs_path.iterdir():
    if file.suffix == '.pdf':
        loader = PyPDFLoader(str(file))
        documents.extend(loader.load())
        print(f"Loaded PDF: {file.name}")
    elif file.suffix == '.txt':
        loader = TextLoader(str(file))
        documents.extend(loader.load())
        print(f"Loaded TXT: {file.name}")

print(f"\nTotal pages/sections loaded: {len(documents)}")
print(f"\nSample of first document:")
print(documents[0].page_content[:500])

Loaded PDF: maintenance_guide.pdf

Total pages/sections loaded: 5

Sample of first document:
Defining And Maintaining Fluid Cleanliness For Maximum Hydraulic Component Life 
 
By Brendan Casey 
 
Many factors can reduce the service life of hydraulic components. Contamination of hydraulic 
fluid by insoluble particles is one of these factors. To prevent particle contamination from 
cutting short component life, an appropriate fluid cleanliness level must first be defined and 
then maintained on a continuous basis.  
 
Particle Contamination And Its Consequences 
 
Particle contamination 


In [2]:
# First try — 500 word chunks with 50 word overlap
text_splitter = RecursiveCharacterTextSplitter (
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print(f"Number of chunks: {len(chunks)}")
print(f"Average chunk size: {sum(len(c.page_content) for c in chunks) / len(chunks):.0f} chars")
print(f"\nSample chunk:")
print(chunks[0].page_content)
print(f"\nChunk metadata: {chunks[0].metadata}")

Number of chunks: 34
Average chunk size: 417 chars

Sample chunk:
Defining And Maintaining Fluid Cleanliness For Maximum Hydraulic Component Life 
 
By Brendan Casey 
 
Many factors can reduce the service life of hydraulic components. Contamination of hydraulic 
fluid by insoluble particles is one of these factors. To prevent particle contamination from 
cutting short component life, an appropriate fluid cleanliness level must first be defined and 
then maintained on a continuous basis.  
 
Particle Contamination And Its Consequences

Chunk metadata: {'producer': 'Acrobat Distiller 5.0.5 (Windows)', 'creator': 'Acrobat PDFMaker 5.0 for Word', 'creationdate': '2003-02-25T12:35:13+08:00', 'author': 'Brendan Casey', 'moddate': '2003-02-25T12:36:08+08:00', 'title': '1', 'source': 'docs/maintenance_guide.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}


In [3]:
# Try smaller chunks — 300 chars
splitter_small = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks_small = splitter_small.split_documents(documents)

# Try larger chunks — 800 chars
splitter_large = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=80)
chunks_large = splitter_large.split_documents(documents)

print(f"Small chunks (300): {len(chunks_small)} chunks")
print(f"Medium chunks (500): {len(chunks)} chunks")
print(f"Large chunks (800): {len(chunks_large)} chunks")

Small chunks (300): 55 chunks
Medium chunks (500): 34 chunks
Large chunks (800): 21 chunks


## Chunking decisions
- Chunk size: 500 characters with 50 character overlap
- Splitter: RecursiveCharacterTextSplitter — splits at paragraph breaks first, then line breaks, then sentences, preserving
  natural language boundaries
- Overlap: 50 characters prevents information loss at chunk boundaries
- Tested 300 and 800 — 300 loses too much context per chunk, 800 reduces retrieval precision
- Total chunks produced: 34

## Embed and store in FAISS

In [4]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Load embedding model
# all-MiniLM-L6-v2: small, fast, good quality — standard default
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")
print(f"Embedding dimension: {len(embedding_model.embed_query('test'))}")

/var/folders/34/_vpjhqmx3t73gmys5kbjy7100000gp/T/ipykernel_19022/4225948326.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
Embedding dimension: 384


In [5]:
# Embed two semantically similar sentences and one different one
sentence_a = "The tool needs replacement after 200 minutes of use."
sentence_b = "Replace the cutting tool when wear exceeds 200 min."
sentence_c = "The weather in Paris is sunny today."

vec_a = embedding_model.embed_query(sentence_a)
vec_b = embedding_model.embed_query(sentence_b)
vec_c = embedding_model.embed_query(sentence_c)

# Compute cosine similarity manually 
import numpy as np

def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

sim_ab = cosine_similarity(vec_a, vec_b)
sim_ac = cosine_similarity(vec_a, vec_c)

print(f"Similarity A-B (same meaning): {sim_ab:.3f}")
print(f"Similarity A-C (different topic): {sim_ac:.3f}")

Similarity A-B (same meaning): 0.669
Similarity A-C (different topic): -0.035


In [6]:
print("Building FAISS index — embedding all chunks...")
vectorstore = FAISS.from_documents(chunks, embedding_model)
print(f"Index built with {vectorstore.index.ntotal} vectors")

# Save locally so you don't re-embed every time
vectorstore.save_local("faiss_index")
print("Index saved to faiss_index/")

Building FAISS index — embedding all chunks...
Index built with 34 vectors
Index saved to faiss_index/


In [7]:
# Load the saved index
vectorstore = FAISS.load_local(
    "faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True
)

# Test with a question
query = "What are the typical internal clearances for servo valves, and why are particles smaller than 5 microns dangerous for hydraulic systems?"
retrieved_chunks = vectorstore.similarity_search(query, k=3)

print(f"Query: {query}\n")
for i, chunk in enumerate(retrieved_chunks):
    print(f"--- Chunk {i+1} ---")
    print(f"Source: {chunk.metadata}")
    print(chunk.page_content)
    print()

Query: What are the typical internal clearances for servo valves, and why are particles smaller than 5 microns dangerous for hydraulic systems?

--- Chunk 1 ---
Source: {'producer': 'Acrobat Distiller 5.0.5 (Windows)', 'creator': 'Acrobat PDFMaker 5.0 for Word', 'creationdate': '2003-02-25T12:35:13+08:00', 'author': 'Brendan Casey', 'moddate': '2003-02-25T12:36:08+08:00', 'title': '1', 'source': 'docs/maintenance_guide.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}
CLEARANCE IN MICRONS 
Gear pump 0.5 – 5.0 
Vane pump 0.5 – 10 
Piston pump 0.5 – 5.0 
Servo valve 1.0 – 4.0 
Control valve 0.5 – 40 
Linear actuator 50 - 250 
 
Particles larger than a component's internal clearances are not necessarily dangerous. Particles 
the same size as the internal clearance cause damage through friction. But the most dangerous 
particles in the long-term are those that are smaller than the component's internal clearances.

--- Chunk 2 ---
Source: {'producer': 'Acrobat Distiller 5.0.5 (Windows)'

## Embedding and retrieval
- Model: sentence-transformers/all-MiniLM-L6-v2
- Embedding dimension: 384
- Verified semantically similar sentences have higher cosine similarity than unrelated ones (A-B: 0.669, A-C: -0.035)
- FAISS index built with 34 vectors, saved locally
- Retrieval test: query "What are the typical internal clearances for servo valves, and why are particles smaller than 5 microns dangerous for hydraulic systems?" retrieved relevant chunks 

## Connect to Gemini and generate answers

In [16]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
import os

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-pro",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0.1
)

prompt = PromptTemplate.from_template("""You are a maintenance engineer assistant.
Answer the question using ONLY the context provided below.
If the answer is not in the context, say "I cannot find this information
in the provided documents" — do not make up an answer.

Context:
{context}

Question: {question}

Answer:""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Chain built successfully")

Chain built successfully


In [17]:
test_questions = [
    "According to Exhibit 1.1, what are the typical internal clearances (in microns) for piston pumps, servo valves, and linear actuators?",
    "Why are particles smaller than 5 microns considered more dangerous for hydraulic systems than larger particles?",
    "For a high-pressure hydraulic system (250–400 bar), what is the minimum recommended cleanliness level in ISO 4406, and what filtration level (in microns) is required to achieve it?",
    "If a fluid sample from a normal-pressure system shows an ISO cleanliness level of 19/16, what steps should be taken to rectify the contamination?",
    "Why is it risky to replace a 20-micron nominal filter with a 10-micron absolute filter without consulting the manufacturer’s pressure drop graphs?"
]

for question in test_questions:
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    answer = qa_chain.invoke(question)
    print(f"A: {answer}")


Q: According to Exhibit 1.1, what are the typical internal clearances (in microns) for piston pumps, servo valves, and linear actuators?


ChatGoogleGenerativeAIError: Error calling model 'gemini-pro' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-pro is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}